# Valoração

Este notebook documenta o passo a passo do processo de valoração da ferramenta de Valoração. O objetivo é criar um pipeline limpo, rastreável e performático para ler, processar e calcular as regras de preços e impostos (N13).

## Importação das Bibliotecas

*   **`pandas`**: Biblioteca principal para manipulação e análise de dados. Será usada para ler as planilhas, realizar os cruzamentos de tabelas (merges) e aplicar as fórmulas de valoração.
*   **`Path` (da biblioteca `pathlib`)**: Facilita a gestão de caminhos de arquivos de forma robusta e multiplataforma (garante que o código funcione sem alterações no Windows, Linux ou macOS).
*   **`time`**: Biblioteca nativa do Python utilizada para medir o tempo de processamento de cada bloco, permitindo identificar gargalos de performance.
*   **`numpy`**: Biblioteca para computação científica. Será extremamente útil para aplicar lógicas condicionais rápidas e gerenciar valores ausentes ou nulos.

In [1]:
import pandas as pd
from pathlib import Path
import time
import numpy as np

## Criação da Base N13P

Nesta etapa, realizamos a leitura otimizada das nossas três bases de dados de entrada: **Ciclo N13P**, **Clientes** e **Produtos**. 

Para maximizar a performance e a segurança do pipeline, aplicamos as seguintes práticas:
*   **Engine Calamine**: Leitura ultra-rápida de planilhas Excel utilizando uma engine baseada em Rust.
*   **Seleção de Colunas**: Importamos apenas as colunas estritamente necessárias para o cálculo de valoração, economizando memória.
*   **Tipagem Estrita**: Forçamos IDs e códigos identificadores como texto (`string`/`object`) para evitar que o Python remova zeros à esquerda.
*   **Precisão Decimal**: Todos os valores numéricos flutuantes (`float`) serão exibidos e tratados com precisão de 4 casas decimais.


In [ ]:
# global setting pandas
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if isinstance(x, (int, float)) else str(x))

period = 3
year = 2026


# path to the files
caminho_dados = Path('../data/')

arquivo_n13p = caminho_dados / f"Ciclo_P{period:02d} N13P {year} - envio.xlsx"
arquivo_clientes = caminho_dados / 'BASE CLIENTES.xlsx'
arquivo_produtos = caminho_dados / 'BASE PRODUTOS.xlsx'

# columns names and types 
colunas_n13p = {
    'Tipo 1': str,
    'Tipo 2': str,
    'Tipo 3': str,
    'Regional': str,
    'GP': str,
    'Vend.': str,
    'Gerente': str,
    'Rede': str,
    'COD_CLIENTE': str,
    'Company Code': str,
    'CD': str,
    'NOME_CLIENTE': str,
    'UF': str,
    'Região': str,
    'EAN': str,
    'SKU': str,
    'Desc. SKU': str,
    'Classificação': str,
    'Tech': str,
    'Tech 2': str,
    'Subbrand': str,
    'Size': str,
    'Nivel 3 HieraR': str,
    'Marca': str
}
colunas_periodos = [f'P{i:02d}-{year}' for i in range(period, 14)]

for col_periodo in colunas_periodos:
    colunas_n13p[col_periodo] = float


colunas_clientes = {
    'COD_CLIENTE': str,
    'NOME_CLIENTE': str,
    'UF': str,
    'Rede': str,
    'COD REDE': str,
    'COD SUBREDE': str,
    'COND. PAG': str,
    'GP': str,
    'COD GP': str
}

colunas_produtos = {
    'EAN': str,
    'Descrição': str,
    'SKU': str,
    'Family Price': str,
    'Categoria': str,
    'Brand': str,
    'Sub Brand': str,
    'Ton/CDA': float,
    'Unid/CX': float, # originalmente é 'Unid/\nCX	': float
    'Origem': str,
    'Hierarquia': str,
    'NCM': str,
    'Tipo': str,
    'Promoção': str,
    'Class.': str,
    'kg/Un': float,
    'H05': str,
    'LSV': float,
}

# read files
start_time = time.time()

df_n13p = pd.read_excel(arquivo_n13p, engine='calamine', header=3, usecols=colunas_n13p.keys(), dtype=colunas_n13p)
df_clientes = pd.read_excel(arquivo_clientes, engine='calamine', header=0, usecols=colunas_clientes.keys(), dtype=colunas_clientes)
df_produtos = pd.read_excel(arquivo_produtos, engine='calamine', header=0, usecols=colunas_produtos.keys(), dtype=colunas_produtos)

end_time = time.time()

print(f"⏱️ Tempo de leitura e tratamento inicial: {end_time - start_time:.4f} segundos")


⏱️ Tempo de leitura e tratamento inicial: 16.4723 segundos


In [12]:
df_produtos.head(10)

,EAN,Descrição,SKU,Family Price,Categoria,Brand,Sub Brand,Ton/CDA,Unid/CX,Origem,Hierarquia,NCM,Tipo,Promoção,Class.,kg/Un,H05,LSV
0,7896029047378,OPT CAT C&T ANTI BOLA PELO 34X80G 2025,60010069,"OPT CAT SCKS 0,008G",CATCARE,CAT SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050204010101,2309.10.00,Regular,0,C&T,0.08000000,0502040101,336.60000000
1,7896029047361,OPT DOG C&T PELE & PELO 34X80G 2025,60010014,"OPT DOG SCKS 0,008G",DOGCARE,DOG SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050104010102,2309.10.00,Regular,0,C&T,0.08000000,0501040101,302.60000000
2,7896029047354,OPT DOG C&T SIS IMUN 34X80G 2025,60010012,"OPT DOG SCKS 0,008G",DOGCARE,DOG SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050104010101,2309.10.00,Regular,0,C&T,0.08000000,0501040101,302.60000000
3,7896029047392,OPT CAT C&T PELE & PELO 34X80G 2025,60010070,"OPT CAT SCKS 0,008G",CATCARE,CAT SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050204010102,2309.10.00,Regular,0,C&T,0.08000000,0502040101,336.60000000
4,7896029047378,OPT CAT C&T ANTI BOLA PELO 34X80G 2025,60010069,"OPT CAT SCKS 0,008G",CATCARE,CAT SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050204010101,2309.10.00,Regular,0,C&T,0.08000000,0502040101,336.60000000
5,7896029047392,OPT CAT C&T PELE & PELO 34X80G 2025,60010070,"OPT CAT SCKS 0,008G",CATCARE,CAT SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050204010102,2309.10.00,Regular,0,C&T,0.08000000,0502040101,336.60000000
6,7896029047361,OPT DOG C&T PELE & PELO 34X80G 2025,60010014,"OPT DOG SCKS 0,008G",DOGCARE,DOG SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050104010102,2309.10.00,Regular,0,C&T,0.08000000,0501040101,302.60000000
7,7896029047354,OPT DOG C&T SIS IMUN 34X80G 2025,60010012,"OPT DOG SCKS 0,008G",DOGCARE,DOG SCKS,OPT SCKS,0.00272000,34.00000000,Nacional,050104010101,2309.10.00,Regular,0,C&T,0.08000000,0501040101,302.60000000
8,7896029047408,"BISCROK RP\nBANANA, LEITE E AVEIA\n22X500G",60010120,"PED BISCROK SCKS 0,5 KG",DOGCARE,DOG SCKS,PED SCKS,0.01100000,22.00000000,Nacional,050104031603,2309.90.30,Regular,0,C&T,0.50000000,0501040316,569.80000000
9,7896029047415,"BISCROK RP\nMAÇA, LINHAÇA E AVEIA\n22X500G",60010122,"PED BISCROK SCKS 0,5 KG",DOGCARE,DOG SCKS,PED SCKS,0.01100000,22.00000000,Nacional,050104031602,2309.90.30,Regular,0,C&T,0.50000000,0501040316,569.80000000
